In [1]:
import dai
import pandas as pd
pd.set_option('display.max_rows', None)

In [2]:
sql = """
WITH 
data_base AS (
    SELECT date
    FROM all_trading_days
    WHERE market_code = 'CN'
    QUALIFY date >= '2022-12-01'
)
SELECT 
    date,
    ROW_NUMBER(date) AS date_index,
FROM data_base
ORDER BY date
"""

df = dai.query(sql).df()

In [3]:
df["dt_year"]      = df["date"].dt.year.astype(str)
df["dt_quarter"]   = df["date"].dt.to_period("Q").apply(lambda x: f"{x.year}{x.quarter:02}")
df["dt_month"]     = df["date"].dt.to_period("M").apply(lambda x: f"{x.year}{x.month:02}")
df["dt_fortnight"] = df["date"].apply(lambda x: f"{x.year}{((x.dayofyear - 1) // 14) + 1:02}")
df["dt_week"]      = df["date"].dt.strftime('%Y%U')
df["dt_day"]       = df["date"].dt.strftime("%Y%m%d")

In [4]:
df["is_year_start_trade"]      = (df["dt_year"] != df["dt_year"].shift(1)).astype(int)
df["is_year_end_trade"]        = (df["dt_year"] != df["dt_year"].shift(-1)).astype(int)

df["is_quarter_start_trade"]   = (df["dt_quarter"] != df["dt_quarter"].shift(1)).astype(int)
df["is_quarter_end_trade"]     = (df["dt_quarter"] != df["dt_quarter"].shift(-1)).astype(int)

df["is_month_start_trade"]     = (df["dt_month"] != df["dt_month"].shift(1)).astype(int)
df["is_month_end_trade"]       = (df["dt_month"] != df["dt_month"].shift(-1)).astype(int)

df["is_fortnight_start_trade"] = (df["dt_fortnight"] != df["dt_fortnight"].shift(1)).astype(int)
df["is_fortnight_end_trade"]   = (df["dt_fortnight"] != df["dt_fortnight"].shift(-1)).astype(int)

df["is_week_start_trade"]      = (df["dt_week"] != df["dt_week"].shift(1)).astype(int)
df["is_week_end_trade"]        = (df["dt_week"] != df["dt_week"].shift(-1)).astype(int)

df["is_year_start_order"]      = df["is_year_start_trade"].shift(-1, fill_value=0)
df["is_year_end_order"]        = df["is_year_end_trade"].shift(-1, fill_value=0)

df["is_quarter_start_order"]   = df["is_quarter_start_trade"].shift(-1, fill_value=0)
df["is_quarter_end_order"]     = df["is_quarter_end_trade"].shift(-1, fill_value=0)

df["is_month_start_order"]     = df["is_month_start_trade"].shift(-1, fill_value=0)
df["is_month_end_order"]       = df["is_month_end_trade"].shift(-1, fill_value=0)

df["is_fortnight_start_order"] = df["is_fortnight_start_trade"].shift(-1, fill_value=0)
df["is_fortnight_end_order"]   = df["is_fortnight_end_trade"].shift(-1, fill_value=0)

df["is_week_start_order"]      = df["is_week_start_trade"].shift(-1, fill_value=0)
df["is_week_end_order"]        = df["is_week_end_trade"].shift(-1, fill_value=0)

In [5]:
df['day_of_quarter_nature_trade']   = df['date'].apply(lambda x: (x - x.to_period('Q').start_time).days + 1)
df['day_of_year_nature_trade']      = df['date'].apply(lambda x: (x - x.to_period('Y').start_time).days + 1)
df['day_of_month_nature_trade']     = df['date'].apply(lambda x: x.day)
df['day_of_fortnight_nature_trade'] = df['date'].apply(lambda x: ((x.dayofyear - 1) % 14) + 1)
df['day_of_week_nature_trade']      = df['date'].dt.dayofweek + 1  # ISO weekday format (1 = Monday, 7 = Sunday)

df['day_of_quarter_nature_order']   = df['day_of_quarter_nature_trade'].shift(-1, fill_value=0)
df['day_of_year_nature_order']      = df['day_of_year_nature_trade'].shift(-1, fill_value=0)
df['day_of_month_nature_order']     = df['day_of_month_nature_trade'].shift(-1, fill_value=0)
df['day_of_fortnight_nature_order'] = df['day_of_fortnight_nature_trade'].shift(-1, fill_value=0)

In [6]:
df["day_of_year_trading_trade"]      = df.groupby("dt_year").cumcount() + 1
df["day_of_quarter_trading_trade"]   = df.groupby("dt_quarter").cumcount() + 1
df["day_of_month_trading_trade"]     = df.groupby("dt_month").cumcount() + 1
df["day_of_fortnight_trading_trade"] = df.groupby("dt_fortnight").cumcount() + 1
df["day_of_week_trading_trade"]      = df.groupby("dt_week").cumcount() + 1

df["day_of_year_trading_order"]      = df["day_of_year_trading_trade"].shift(-1, fill_value=0)
df["day_of_quarter_trading_order"]   = df["day_of_quarter_trading_trade"].shift(-1, fill_value=0)
df["day_of_month_trading_order"]     = df["day_of_month_trading_trade"].shift(-1, fill_value=0)
df["day_of_fortnight_trading_order"] = df["day_of_fortnight_trading_trade"].shift(-1, fill_value=0)
df["day_of_week_trading_order"]      = df["day_of_week_trading_trade"].shift(-1, fill_value=0)

In [7]:
df

,date,date_index,dt_year,dt_quarter,dt_month,dt_fortnight,dt_week,dt_day,is_year_start_trade,is_year_end_trade,...,day_of_year_trading_trade,day_of_quarter_trading_trade,day_of_month_trading_trade,day_of_fortnight_trading_trade,day_of_week_trading_trade,day_of_year_trading_order,day_of_quarter_trading_order,day_of_month_trading_order,day_of_fortnight_trading_order,day_of_week_trading_order
0,2022-12-01,1,2022,202204,202212,202224,202248,20221201,1,0,...,1,1,1,1,1,2,2,2,2,2
1,2022-12-02,2,2022,202204,202212,202224,202248,20221202,0,0,...,2,2,2,2,2,3,3,3,1,1
2,2022-12-05,3,2022,202204,202212,202225,202249,20221205,0,0,...,3,3,3,1,1,4,4,4,2,2
3,2022-12-06,4,2022,202204,202212,202225,202249,20221206,0,0,...,4,4,4,2,2,5,5,5,3,3
4,2022-12-07,5,2022,202204,202212,202225,202249,20221207,0,0,...,5,5,5,3,3,6,6,6,4,4
5,2022-12-08,6,2022,202204,202212,202225,202249,20221208,0,0,...,6,6,6,4,4,7,7,7,5,5
6,2022-12-09,7,2022,202204,202212,202225,202249,20221209,0,0,...,7,7,7,5,5,8,8,8,6,1
7,2022-12-12,8,2022,202204,202212,202225,202250,20221212,0,0,...,8,8,8,6,1,9,9,9,7,2
8,2022-12-13,9,2022,202204,202212,202225,202250,20221213,0,0,...,9,9,9,7,2,10,10,10,8,3
9,2022-12-14,10,2022,202204,202212,202225,202250,20221214,0,0,...,10,10,10,8,3,11,11,11,9,4
